In [ ]:

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
def space():
    print("- -" * 50)

In [ ]:

df = pd.read_csv("data-raw/SA_Aqar.csv", encoding="utf-8-sig")


In [ ]:
df.head(1)


In [ ]:

print(df.shape)
space()
print(df.dtypes)
space()
print(df.columns)

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


In [ ]:
print(df.duplicated())
df.tail(10)

In [ ]:
print(df.shape)

In [ ]:
counts = df.value_counts(dropna=False).reset_index(name="copies")
groups = counts[counts["copies"] > 1]
print(len(groups), "listings have copies")
groups.drop(columns="details")


In [ ]:
df =df.drop_duplicates()

In [ ]:
df.shape

### Why not a normal random split
The usual way is train_test_split(df, test_size=0.2, random_state=42) from scikit-learn. Its random_state also makes it repeatable, but it has two weaknesses here:

Copies can be split up. It decides row by row, so one copy of a house can land in train and the other in test. crc32 gives copies the same number, so they always land together.
Removing rows reshuffles everything. If you later drop rows, for example the near-copies or the 40 houses in the app's sample data, a seeded shuffle hides a different set of houses. With crc32, each house's place depends only on its own facts, so the houses that remain stay where they were.

In [ ]:
import zlib

FACTS = df.columns.drop(["price", "details"]).tolist()             # every column except price and details
key = df[FACTS].astype(str).agg("|".join, axis=1)                  # one long text per house, made from its facts
df["is_test"] = key.map(lambda k: zlib.crc32(k.encode()) % 5 == 0)  # about 1 house in 5 gets True = hidden

df["is_test"].value_counts()


In [ ]:
train = df[~df["is_test"]]  # ~ means "not": the 1,197 houses that aren't hidden
